# Principia V1.3 Tutorial

This notebook demonstrates the standard real-LLM Principia V1.3 workflow.

The core stages are intentionally separate:

1. **Research**: retrieve real works from public metadata sources.
2. **Information extraction**: extract structured features from retrieved works with a real LLM.
3. **Evidence selection**: choose which extracted ideas, principles, takeaways, baselines, benchmarks, and result facts should feed generation.
4. **Generate idea**: synthesize one evidence-grounded idea with a real LLM.
5. **Compare**: check the generated idea against prior ideas extracted from the retrieved works.


## Installation And Kernel Setup

Install Principia before running this notebook:

```bash
pip install principia-ai ipykernel
```

For local source development from the package repository:

```bash
pip install -e . ipykernel
```

For VS Code, create and select a dedicated notebook kernel:

```bash
python -m ipykernel install --user --name principia-v13-python --display-name "Python 3.12 (Principia V1.3)"
```

Then open this notebook and select `Python 3.12 (Principia V1.3)`. If VS Code does not show the named kernel, select the Python interpreter from the virtual environment where `principia-ai` and `ipykernel` were installed.

This tutorial is designed for real LLM execution only.


## 1. Setup

Set `SILICONFLOW_API_KEY` in your shell before launching the notebook, or replace `YOUR_SILICONFLOW_API_KEY` in the setup cell for a local run.

The same `WORKSPACE_DIR` is used throughout the notebook so search, extraction, resume, and export all read the same project state.


In [ ]:
import os

import principia as pc
from IPython.display import Markdown, display

API_key = os.environ.get("SILICONFLOW_API_KEY", "YOUR_SILICONFLOW_API_KEY")
WORKSPACE_DIR = "principia_astro"

EXTRACT_MODEL = "siliconflow:Qwen/Qwen3.6-27B"
IDEA_MODEL = "siliconflow:Deepseek/DeepSeek-V4-Pro"

goal = "Develop theory-driven, evidence-grounded hypotheses to explain possible electromagnetic emission associated with S251112cm, a sub-solar-mass compact-object merger candidate. Starting from reported optical transient candidates and existing follow-up constraints, synthesize principles from kilonovae, super-kilonovae, kilonovae-within-supernovae, AGN-disk merger flares, and ordinary supernova contaminants, and infer which physical scenarios can consistently explain the observed temporal, spatial, photometric, spectroscopic, and host-galaxy evidence."

pc.__version__


Expected output: `1.3.2`.

## 2. Create Workspace

The workspace stores user-facing outputs in visible files and internal state in `.principia/`:

```text
principia_astro/
  README.md
  principia_outputs/
    latest/
      idea.md
      result.json
      works.json
    exports/
      <idea_id>/
        idea.md
        result.json
        works.json
  .principia/
    principia.sqlite
    artifacts/
      source_json/
      exports/
      pdfs/
      cache/
```


In [ ]:


ws = pc.Workspace(
    WORKSPACE_DIR,
    llm_config=pc.siliconflow_config(API_key, timeout=900, max_retries=4),
)

ws.path


## 3. Research: Search Real Works

This cell retrieves 50 real works from public metadata sources and stores them in SQLite.

Principia now uses the shared semantic retriever over arXiv, OpenAlex, Crossref, and Semantic Scholar. It plans multiple academic search queries from the goal, deduplicates candidates by DOI/arXiv/OpenAlex/Semantic Scholar IDs and normalized title, and uses LLM semantic reranking when a real LLM is configured. Without an LLM it falls back to deterministic semantic-lite ranking.

For non-AI goals, AI/LLM/MAS query expansion is not triggered unless the goal explicitly asks for AI, LLMs, agents, or MAS.

The progress display is provided by Principia. It includes elapsed time and ETA.


In [ ]:

search_progress = pc.notebook_progress("Research search")

works = ws.research.search(
    goal,
    target_count=50,
    callback=search_progress,
)

len(works), works[0].title


## 4. Review Retrieved Works

A `WorkItem` stores metadata such as `id`, `title`, `authors`, `abstract`, `year`, `venue`, `url`, `doi`, `arxiv_id`, and `openalex_id`.

The table below shows the top 12 retrieved works. Re-run the research cell after changing the goal or retriever code so the displayed works reflect the current search implementation.


In [ ]:
work_rows = [
    [i + 1, w.title, pc.work_review_status(w), w.year or "", w.venue or w.source, w.url]
    for i, w in enumerate(works)
]
display(Markdown(pc.markdown_table(["#", "Title", "Review status", "Year", "Venue/Source", "URL"], work_rows)))


## 5. Information Extraction

This cell extracts structured research features from the top-ranked works using DeepSeek.

For a tutorial, extracting the top 20 works keeps runtime and provider timeout risk manageable while preserving the full 50-work research list. Increase `EXTRACT_COUNT` and `max_chars` after the first run succeeds.

The progress display includes ETA and per-item counts. During a provider call, Principia keeps the display alive with heartbeat updates rather than freezing on a fixed percentage.


In [ ]:

EXTRACT_COUNT = 20
extract_progress = pc.notebook_progress("Information extraction")

features = ws.research.extract(
    works[:EXTRACT_COUNT],
    model=EXTRACT_MODEL,
    overwrite=False,
    max_chars=12_000,
    continue_on_error=True,
    callback=extract_progress,
)

features.counts(), features.model


## 6. Review Extracted Features

`ExtractedFeatures` is a batch object. Each item is a `WorkFeatures` record with these feature buckets:

- `ideas`: prior or existed ideas extracted from the work.
- `principles`: reusable principles or mechanisms.
- `takeaways`: actionable lessons or result messages.
- `baselines`: comparison methods.
- `benchmarks`: datasets, tasks, or evaluation settings.
- `result_facts`: grounded factual results.

Each extracted record has an `id`. You can use those IDs in the next step to select specific evidence for idea generation.


In [ ]:

display(Markdown(pc.feature_summary_markdown(features, limit=8)))

## Optional: Start A New Notebook From Existing Features

If you already completed research and information extraction in this workspace, a new notebook can resume from persisted SQLite records.

Use the same workspace folder and call `ws.load_features()`. This does not rerun public search or LLM extraction.


In [ ]:
resumed_ws = pc.Workspace(
    WORKSPACE_DIR,
    llm_config=pc.siliconflow_config(API_key, timeout=900, max_retries=4),
)

resumed_features = resumed_ws.load_features()
resumed_features.counts()


## 7. Select Evidence For Idea Generation

By default, `pc.select_evidence(features)` selects all extracted feature buckets.

You can narrow the generation input with:

- `kinds=["ideas", "principles", "takeaways"]`
- `work_ids=[...]`
- `feature_ids=[...]`
- `limit_per_kind=...`

The result is an `EvidencePacket`, which is the direct input to `ws.ideas.generate(...)`.


In [ ]:
selected_evidence = pc.select_evidence(
    features,
    kinds=["ideas", "principles", "takeaways", "baselines", "benchmarks", "result_facts"],
)

selected_evidence.counts()


## 8. Generate Idea

This cell generates one evidence-grounded idea from `selected_evidence` using Qwen.

The generated `Idea` includes thesis, novelty claim, mechanistic design, methodological details, method variants, validation protocol, metrics, risks, assumptions, source evidence, lineage or trace metadata, and generation metadata.


In [ ]:

generate_progress = pc.notebook_progress("Idea generation")

idea = ws.ideas.generate(
    selected_evidence,
    user_note=goal,
    mode="calculus",
    model=IDEA_MODEL,
    callback=generate_progress,
)

idea.title


## 9. Review Generated Idea

`pc.idea_markdown(...)` renders the full idea card, including Methodological Details and LaTeX equations when present.


In [ ]:

display(Markdown(pc.idea_markdown(idea)))


## 10. Data Structure Reference

The tables below show the public fields available on the main returned objects. The full API reference is in `docs/api.md`.


In [ ]:

display(Markdown("### Idea fields\n" + pc.schema_markdown(pc.Idea)))
display(Markdown("### WorkFeatures fields\n" + pc.schema_markdown(pc.WorkFeatures)))
display(Markdown("### EvidencePacket fields\n" + pc.schema_markdown(pc.EvidencePacket)))


## 11. Compare Against Prior Ideas

This optional final step compares the generated idea with ideas extracted from the retrieved works.

The comparison progress display also uses heartbeat updates during long provider calls.


In [ ]:

compare_progress = pc.notebook_progress("Idea comparison")

comparison = ws.ideas.compare(
    idea,
    features,
    model=IDEA_MODEL,
    callback=compare_progress,
)

len(comparison.rows)


## 12. Review Comparison

In [ ]:

comparison_rows = [
    [
        row.get("title", ""),
        row.get("mechanistic_similarity", ""),
        row.get("essential_difference", ""),
        row.get("potential_advantage", ""),
    ]
    for row in comparison.rows[:8]
]

display(Markdown(pc.markdown_table(["Prior work", "Similarity", "Difference", "Advantage"], comparison_rows)))


## 13. Local Outputs

Export the staged workflow result to local files.

The canonical export remains under hidden `.principia/artifacts/exports/`, and a visible mirror is written to:

```text
principia_astro/principia_outputs/latest/
  idea.md
  result.json
  works.json
```

The exported `idea.md` includes Methodological Details, formulas, validation plan, risks, source evidence, and comparison highlights.

The database counts show what is stored in SQLite. Rerun with `overwrite=True` only when you intentionally want to redo completed LLM extraction work.



In [ ]:
export_path = ws.export(
    goal=goal,
    works=works,
    features=features,
    idea=idea,
    comparison=comparison,
)

ws.counts(), export_path, ws.outputs_dir / "latest"
